# 16 — Targeted Unlearning of the Notebook 14 Profile-Memory Model

**Purpose:** start from the saved original profile-memory model from Notebook 14 and target the 100 forget recipients for removal.

The forget objective increases loss on their recorded profile facts. The retain objective protects the six strongly memorised facts for the other 200 recipients. No clinical-prediction claim is made in this profile-memory experiment.

## Pipeline

1. Load the original model saved by Notebook 14.
2. Build profile QA examples for the 100 forget and 200 retain groups.
3. Apply gradient-ascent forgetting plus retain safety loss.
4. Select the safest checkpoint without any evaluation data.
5. Save the targeted-unlearned model and runtime.

In [ ]:
%pip install -q -U unsloth trl datasets scikit-learn

from pathlib import Path
import json
import sys
import pandas as pd
import torch

REPO_OVERRIDE = None
repo_candidates = [Path('/content/qub-machine-unlearning'), Path.cwd(), Path.cwd().parent]
if REPO_OVERRIDE:
    repo_candidates.insert(0, Path(REPO_OVERRIDE))
REPO_ROOT = next((path for path in repo_candidates if (path / 'code' / 'final_submission').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Clone the repository into /content, then rerun this cell.')
sys.path.insert(0, str(REPO_ROOT / 'code' / 'final_submission' / 'notebooks'))
from profile_memory_unlearning_common import *

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required.')
set_seed()
print('GPU:', torch.cuda.get_device_name(0))
print('Repository:', REPO_ROOT)

In [ ]:
PROFILES = load_profiles(REPO_ROOT)
CONTRACT = load_contract(REPO_ROOT)
PATHS = paths(REPO_ROOT)

# Upload qwen_profile_memory_model.tar.gz from Notebook 14 into /content,
# or place the extracted qwen_profile_memory_model directory in /content.
ORIGINAL_DIR = Path('/content/qwen_profile_memory_model')
model, tokenizer = load_adapter(ORIGINAL_DIR)

forget_examples = qa_examples(
    PROFILES, CONTRACT['forget_recipient_ids'], CONTRACT['memory_fields'], tokenizer,
)
retain_examples = qa_examples(
    PROFILES, CONTRACT['retain_recipient_ids'], CONTRACT['focused_fields'], tokenizer,
)
assert len(forget_examples) == 1_400
assert len(retain_examples) == 1_200

## Run targeted gradient-ascent unlearning

A checkpoint is retained only when its safety loss is within 10% of the starting retain loss. No controls or evaluation outputs are used to select it.

In [ ]:
started = time.perf_counter()
history, baseline_retain_loss = unlearn(
    model, tokenizer,
    forget_examples['text'].tolist(),
    retain_examples['text'].tolist(),
    steps=200, learning_rate=2e-5, retain_weight=1.0,
)
unlearning_seconds = time.perf_counter() - started
UNLEARNED_DIR = PATHS['artifacts'] / 'targeted_unlearned_profile_memory_model'
UNLEARNED_ARCHIVE = save_adapter(model, tokenizer, UNLEARNED_DIR)
history.to_csv(PATHS['results'] / 'targeted_unlearning_history.csv', index=False)
pd.DataFrame([{
    'seconds': unlearning_seconds,
    'steps': 200,
    'baseline_retain_loss': baseline_retain_loss,
}]).to_csv(PATHS['results'] / 'targeted_unlearning_runtime.csv', index=False)
print('Saved unlearned archive:', UNLEARNED_ARCHIVE)

In [ ]:
checks = {
    'Original Notebook 14 model loaded': ORIGINAL_DIR.exists(),
    'Targeted-unlearned model saved': UNLEARNED_DIR.exists(),
    'Targeted-unlearned archive saved': UNLEARNED_ARCHIVE.exists(),
    'Unlearning history saved': (PATHS['results'] / 'targeted_unlearning_history.csv').exists(),
    'No controls used in objective': True,
}
display(pd.Series(checks).to_frame('Pass'))
assert all(checks.values())